# Train DistilBERT on Colab (free GPU)

This notebook fine-tunes DistilBERT for the AI Health Assistant using the code in `src/bert_model.py`.
It uses exactly the same train / validation / test splits and labels as the baseline.

**Steps**
1. Check the GPU
2. Get the code from GitHub
3. Install the requirements
4. Quick smoke test (about 1 minute)
5. Train with learning rate 5e-5, then 3e-5 - validation only, test set untouched
6. Pick the winner by **validation** macro-F1
7. Evaluate the winner **once** on the test set
8. Zip `models/bert` and `reports/` and download them

> Educational project only - not a medical diagnosis tool.

## Step 1 - Check the GPU

In the menu choose **Runtime -> Change runtime type -> T4 GPU**, then run the cell below.
You should see a Tesla T4 and `CUDA available: True`. If it says `False`, training still works but will be much slower.

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())

## Step 2 - Get the code

Clone the project repository and move into it. All later commands run from the project folder.

In [ ]:
!git clone https://github.com/zaima-20/ai-health-assistant.git
%cd ai-health-assistant
!ls

## Step 3 - Install the requirements

Colab already has PyTorch and most libraries, so this is usually quick.

In [ ]:
!pip install -q -r requirements.txt

## Step 4 - Smoke test (optional, about 1 minute)

Trains for 1 epoch on 40 rows to check that everything runs. It does **not** use the test set
and does **not** write anything to `models/` or `reports/`.

In [ ]:
!python -m src.bert_model --smoke

## Step 5a - Train with learning rate 5e-5 (validation only)

Up to 8 epochs, with early stopping on validation macro-F1 (patience 2). The best epoch is kept.
`--skip-test` means the test set is **not touched** here.
The model and validation results are copied into `runs/lr_5e-5/` so the next run does not overwrite them.

In [ ]:
!python -m src.bert_model --lr 5e-5 --skip-test
!rm -rf runs/lr_5e-5 && mkdir -p runs/lr_5e-5 && cp -r models/bert runs/lr_5e-5/ && cp reports/bert_metrics.json reports/bert_confusion_val.png runs/lr_5e-5/

## Step 5b - Train with learning rate 3e-5 (validation only)

In [ ]:
!python -m src.bert_model --lr 3e-5 --skip-test
!rm -rf runs/lr_3e-5 && mkdir -p runs/lr_3e-5 && cp -r models/bert runs/lr_3e-5/ && cp reports/bert_metrics.json reports/bert_confusion_val.png runs/lr_3e-5/

## Step 6 - Pick the winner by validation macro-F1

Only validation scores are used - neither run has seen the test set yet.
The winner's model goes to `models/bert/` and its validation results to `reports/`.
Both runs' validation metrics are also kept as `reports/bert_metrics_lr_<lr>.json`.

In [ ]:
import json, shutil
from pathlib import Path

runs = {lr: json.loads(Path(f"runs/lr_{lr}/bert_metrics.json").read_text()) for lr in ["5e-5", "3e-5"]}

print(f"{'lr':<7}{'best epoch':>11}{'epochs run':>11}{'secs':>8}{'val acc':>9}{'val F1':>8}")
for lr, m in runs.items():
    print(f"{lr:<7}{m['best_epoch']:>11}{m['epochs_run']:>11}{m['train_seconds']:>8.0f}"
          f"{m['validation']['accuracy']:>9.4f}{m['validation']['macro_f1']:>8.4f}")

best_lr = max(runs, key=lambda lr: runs[lr]["validation"]["macro_f1"])  # ties -> 5e-5
print(f"
Winner by validation macro-F1: lr={best_lr}")

shutil.rmtree("models/bert", ignore_errors=True)
shutil.copytree(f"runs/lr_{best_lr}/bert", "models/bert")
shutil.copy(f"runs/lr_{best_lr}/bert_metrics.json", "reports/bert_metrics.json")
shutil.copy(f"runs/lr_{best_lr}/bert_confusion_val.png", "reports/bert_confusion_val.png")
for lr in runs:
    shutil.copy(f"runs/lr_{lr}/bert_metrics.json", f"reports/bert_metrics_lr_{lr}.json")

print("
Per-epoch history of the winner:")
for row in runs[best_lr]["history"]:
    print(row)

## Step 7 - Evaluate the winner ONCE on the test set

Loads `models/bert/` and adds the test results to `reports/bert_metrics.json`.
It refuses to run a second time on the same metrics file, so the test set is used once.

In [ ]:
!python -m src.bert_model --test-only

In [ ]:
m = json.loads(Path("reports/bert_metrics.json").read_text())
print(f"lr={m['params']['lr']}  best epoch={m['best_epoch']}  training time={m['train_seconds']} s")
for split in ("validation", "test"):
    print(f"{split:<11} accuracy={m[split]['accuracy']:.4f}  macro-F1={m[split]['macro_f1']:.4f}  (n={m[split]['n_samples']})")

## Step 8 - Zip and download

Downloads `bert_results.zip` with `models/bert/` (about 250 MB) and `reports/`.
On your laptop, unzip it inside the project folder so the files land in `models/bert/` and `reports/`.

In [ ]:
!zip -qr bert_results.zip models/bert reports
!ls -lh bert_results.zip
from google.colab import files
files.download("bert_results.zip")